# 365 Probabilidades - Dia #167
## Qual a probabilidade de um estudo dizer "causa" quando só viu as coisas juntas?

**Tipo:** Curiosidade · Método científico  
**Data de publicação:** 2026-11-27  
**Ferramenta:** Python  
**Decisão analisada:** Quanto confiar numa notícia que diz que uma coisa causa outra?  
**Hashtag:** #365Probabilidades #Dia167

---

### 📖 A História

Quem toma café vive mais. Quem lê antes de dormir dorme melhor. Quem tem cachorro é mais feliz.

Frases assim aparecem todo dia, e quase sempre vêm com um "segundo um estudo". Muitas vezes o estudo só observou que as duas coisas andam juntas, sem testar se uma provoca a outra.

Aqui no 365 Probabilidades, separar associação de causa é uma regra de todo dia. Fui ver com que frequência os próprios cientistas pulam essa linha.

---

### 📚 O Conceito

Um estudo **transversal** observa as pessoas num único momento. Ele pode mostrar que quem toma mais café vive mais, mas não pode mostrar que é o café que faz viver mais. Quem toma café pode ter outros hábitos, outra renda, outro trabalho.

Para falar em causa, é preciso mais. Um sorteio, como num ensaio clínico. Ou um acompanhamento ao longo do tempo com controles fortes. Ou um experimento natural.

Quando um estudo transversal usa verbos como "aumenta", "reduz", "provoca" ou "melhora", ele diz mais do que o desenho permite. Os autores do estudo do dia chamam isso de **afirmação causal exagerada**.

---

### 🧮 O Modelo

Uma fonte principal e seis contas. A conta central é a sexta, uma simulação de Monte Carlo do caminho que um estudo faz até chegar a quem lê.

1. **Quantos exageram.** A proporção de artigos transversais com linguagem causal no título ou no resumo, com intervalo de Jeffreys.
2. **Descontando o erro da máquina.** A mesma proporção corrigida pelos acertos e erros do classificador automático, medidos contra codificação humana (correção de Rogan-Gladen, 200 mil sorteios, semente 42).
3. **Revistas de mais prestígio.** A diferença entre revistas acima e abaixo da mediana de impacto.
4. **Dentro do artigo.** Quantas afirmações do texto completo são causais e sem ressalva.
5. **Quando a IA resume.** Quantos resumos feitos por IA usam linguagem causal, conforme o que se diz ao modelo.
6. **O telefone sem fio (modelo central).** Um estudo transversal passa por dois elos antes de chegar a você, o resumo dos autores e o resumo de uma IA. A probabilidade de ele chegar dizendo "causa" é

   P(chega como causa) = π · q₁ + (1 − π) · q₀

   em que π é a proporção de resumos com linguagem de causa (já corrigida pelo erro do classificador, Conta 2), q₁ é a chance de a IA manter a causa quando o resumo já diz causa e q₀ é a chance de a IA pôr causa num resumo que não diz. Cada termo entra com sua incerteza (Beta de Jeffreys), e a conta roda 20 mil vezes (semente 42). O mesmo cálculo é refeito com a IA recebendo uma instrução de cuidado.

**Fonte:**
- Isch, C., Dörr, T., Fasching, N., Jennings, G. & Watts, D. J. (2026). Quantifying the prevalence and impact of overreaching causal claims in social science. *Nature Human Behaviour*. **194.631 artigos transversais** de ciências sociais publicados entre 1980 e 2024, um experimento com **1.105 leitores** e dois experimentos com resumos feitos por IA. Acesso aberto. Lido na versão disponibilizada pelo primeiro autor.

**Nota metodológica sobre o fator ×0,80.** Não se aplica. A fonte é uma análise automática de textos validada por codificação humana, e não autorrelato.

**Cordas declaradas:** #162 (friagem e resfriado), em que a diferença entre associação e causa foi o centro do texto.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

SEMENTE = 42
N_SIM = 200_000
rng = np.random.default_rng(SEMENTE)

VERMELHO = '#c0392b'
DOURADO = '#c8a84b'
VERDE = '#1a5f5a'
CINZA = '#7f8c8d'

def jeffreys(k, n):
    # intervalo de Jeffreys (IC 95%) para uma proporção k/n
    return stats.beta.ppf([0.025, 0.975], k + 0.5, n - k + 0.5)

def sorteia(k, n):
    # sorteios da proporção k/n pela Beta de Jeffreys
    return rng.beta(k + 0.5, n - k + 0.5, N_SIM)

print("✅ Bibliotecas carregadas")

In [ ]:
# --- DADOS DA LITERATURA ---
# Isch, Dörr, Fasching, Jennings & Watts (2026), Nature Human Behaviour. Texto completo (versão do autor).

n_artigos = 194_631                     # artigos transversais classificados, 1980 a 2024
p_causal = 0.463                        # com linguagem causal no título ou resumo
tendencia = {'até 2000': 0.20, '2024': 0.60}   # "estável em 20%" e "mais de 60% em 2024" (texto)

# Classificador final (BERT) de linguagem causal, testado em 6.027 artigos separados
clf = dict(n_teste=6027, precisao=0.84, sensibilidade=0.82)

# Revistas acima e abaixo da mediana de fator de impacto (3,03), 127.598 artigos com o índice
revistas = dict(n=127_598, acima=0.544, abaixo=0.433)

# Texto completo
texto = dict(n_artigos=69_580, com_trecho_causal=0.818,
             afirmacoes=3134, associativas=2287, causais_fortes=487, causais_condicionais=360)

# Resumos feitos por IA (GPT-4.1), experimento 1: linguagem causal no resumo, conforme o texto de entrada
resumo_ia = {'Recebeu o resumo original': 0.993, 'Recebeu o resumo reescrito sem causa': 0.898,
             'Recebeu o original + aviso do desenho': 0.417, 'Recebeu o original + aviso + crítica': 0.303}
prompt_cuidadoso = 0.150                # experimento 2, média com a instrução de cuidado
n_artigos_exp1 = 28                     # resumos usados no experimento 1 (10 resumos de IA por artigo e condição)
n_artigos_exp2 = 100                    # artigos usados no experimento 2 (cinco modelos)

# Experimento com leitores
leitores = dict(n=1105, or_reescrito=0.38)   # chance de o leitor usar causa no próprio resumo

aplica_fator_080 = False

fmt = lambda x, c=1: f"{x:.{c}f}".replace('.', ',')
mil = lambda x: f"{x:,}".replace(',', '.')
print("=" * 72)
print("  DADOS DA LITERATURA - O ESTUDO QUE DIZ MAIS DO QUE PODE")
print("=" * 72)
print(f"\n  Isch et al. (2026): {mil(n_artigos)} artigos transversais, 1980 a 2024")
print(f"  → Com linguagem causal: {fmt(p_causal*100)}%  ·  até 2000 ~{fmt(tendencia['até 2000']*100, 0)}%  ·  2024 >{fmt(tendencia['2024']*100, 0)}%")
print(f"  → Classificador: precisão {fmt(clf['precisao'], 2)}, sensibilidade {fmt(clf['sensibilidade'], 2)} ({mil(clf['n_teste'])} artigos de teste)")
print(f"  → Revistas acima da mediana de impacto: {fmt(revistas['acima']*100)}%  ·  abaixo: {fmt(revistas['abaixo']*100)}%")
print(f"  → Texto completo ({mil(texto['n_artigos'])} artigos): {fmt(texto['com_trecho_causal']*100)}% com algum trecho causal")
print(f"  → Afirmações do texto (100 artigos, {mil(texto['afirmacoes'])}): {texto['associativas']} associativas, {texto['causais_fortes']} causais sem ressalva, {texto['causais_condicionais']} causais com ressalva")
for k, v in resumo_ia.items():
    print(f"  → Resumo da IA, entrada '{k}': {fmt(v*100)}% com linguagem causal")
print(f"  → Com instrução de cuidado (cinco modelos): {fmt(prompt_cuidadoso*100)}%")
print(f"  → Experimento com leitores: N = {mil(leitores['n'])}")
print(f"\n  Fator ×0,80 aplicado: {'sim' if aplica_fator_080 else 'não'}")
print("=" * 72)

In [ ]:
# --- O MODELO ---

# Conta 1 · Quantos exageram
k_causal = int(round(p_causal * n_artigos))
ic_obs = jeffreys(k_causal, n_artigos)

# Conta 2 · Descontando o erro da máquina (Rogan-Gladen)
# Da precisão P e da sensibilidade S tiramos a especificidade E implícita na proporção observada.
# Prevalência verdadeira: pi = (p_obs + E - 1) / (S + E - 1)
def corrige(p_obs, P, S):
    tp = P * p_obs                      # verdadeiros positivos (fração do total)
    pi = tp / S                         # prevalência verdadeira implícita
    fp = p_obs - tp
    E = 1 - fp / (1 - pi)
    return (p_obs + E - 1) / (S + E - 1), E
pi_central, esp = corrige(p_causal, clf['precisao'], clf['sensibilidade'])
# Incerteza: precisão e sensibilidade sorteadas com o tamanho do conjunto de teste
n_pos_prev = int(clf['n_teste'] * p_causal)
P_sim = rng.beta(clf['precisao'] * n_pos_prev + 0.5, (1 - clf['precisao']) * n_pos_prev + 0.5, N_SIM)
S_sim = rng.beta(clf['sensibilidade'] * n_pos_prev + 0.5, (1 - clf['sensibilidade']) * n_pos_prev + 0.5, N_SIM)
p_sim = rng.beta(k_causal + 0.5, n_artigos - k_causal + 0.5, N_SIM)
pi_sim = np.array([corrige(p, P, S)[0] for p, P, S in zip(p_sim[:20000], P_sim[:20000], S_sim[:20000])])
ic_pi = np.percentile(pi_sim, [2.5, 97.5])
p_mais_de_40 = (pi_sim > 0.40).mean()

# Conta 3 · Revistas de mais prestígio (metade dos artigos de cada lado da mediana)
n_lado = revistas['n'] // 2
sa = sorteia(int(round(revistas['acima'] * n_lado)), n_lado)
sb = sorteia(int(round(revistas['abaixo'] * n_lado)), n_lado)
dif_rev = np.percentile((sa - sb) * 100, [2.5, 50, 97.5])

# Conta 4 · Dentro do artigo
p_forte = texto['causais_fortes'] / texto['afirmacoes']
ic_forte = jeffreys(texto['causais_fortes'], texto['afirmacoes'])
p_assoc = texto['associativas'] / texto['afirmacoes']

# Conta 5 · Quando a IA resume
queda_aviso = (resumo_ia['Recebeu o resumo original'] - resumo_ia['Recebeu o original + aviso do desenho']) * 100

# Conta 6 · O telefone sem fio (Monte Carlo)
# Incerteza contada por artigo (28 no experimento 1, 100 no experimento 2), e não por resumo de IA,
# porque os 10 resumos de um mesmo artigo não são independentes.
N_CADEIA = len(pi_sim)
def beta_prop(p, n, tam):
    return rng.beta(p * n + 0.5, (1 - p) * n + 0.5, tam)
q1_sim = beta_prop(resumo_ia['Recebeu o resumo original'], n_artigos_exp1, N_CADEIA)
q0_sim = beta_prop(resumo_ia['Recebeu o resumo reescrito sem causa'], n_artigos_exp1, N_CADEIA)
cuidado_sim = beta_prop(prompt_cuidadoso, n_artigos_exp2, N_CADEIA)

cadeia_ia = pi_sim * q1_sim + (1 - pi_sim) * q0_sim
ic_cadeia = np.percentile(cadeia_ia, [2.5, 50, 97.5])
ic_cuidado = np.percentile(cuidado_sim, [2.5, 50, 97.5])
p_cadeia_maior_80 = (cadeia_ia > 0.80).mean()
# dos que chegam como causa, quantos tinham resumo que NÃO dizia causa (a IA acrescentou)
frac_acrescentada = np.percentile((1 - pi_sim) * q0_sim / cadeia_ia, [2.5, 50, 97.5])

# Conferência por simulação direta: 100 mil estudos, um a um
n_estudos = 100_000
resumo_causa = rng.random(n_estudos) < pi_central
ia_causa = np.where(resumo_causa, rng.random(n_estudos) < resumo_ia['Recebeu o resumo original'],
                    rng.random(n_estudos) < resumo_ia['Recebeu o resumo reescrito sem causa'])
direto = ia_causa.mean()

print("=" * 72)
print("  MODELO - RESULTADO")
print("=" * 72)
print("\n  CONTA 1 · Quantos exageram")
print(f"  → {mil(k_causal)} de {mil(n_artigos)} = {fmt(p_causal*100)}% [{fmt(ic_obs[0]*100, 2)}; {fmt(ic_obs[1]*100, 2)}]")
print("\n  CONTA 2 · Descontando o erro da máquina")
print(f"  → Especificidade implícita do classificador: {fmt(esp*100)}%")
print(f"  → Proporção corrigida: {fmt(pi_central*100)}% [{fmt(ic_pi[0]*100)}; {fmt(ic_pi[1]*100)}]  ·  P(mais de 40%) {'acima de 99%' if p_mais_de_40 > 0.995 else '= ' + fmt(p_mais_de_40*100, 0) + '%'}")
print("\n  CONTA 3 · Revistas de mais prestígio")
print(f"  → Acima da mediana de impacto, {fmt(dif_rev[1])} pp a mais [{fmt(dif_rev[0])}; {fmt(dif_rev[2])}]")
print("\n  CONTA 4 · Dentro do artigo")
print(f"  → Afirmações causais sem ressalva: {fmt(p_forte*100)}% [{fmt(ic_forte[0]*100)}; {fmt(ic_forte[1]*100)}]  ·  associativas: {fmt(p_assoc*100)}%")
print(f"  → Artigos com ao menos um trecho causal no texto completo: {fmt(texto['com_trecho_causal']*100)}%")
print("\n  CONTA 5 · Quando a IA resume")
print(f"  → Um aviso sobre o desenho do estudo derruba a linguagem causal dos resumos em {fmt(queda_aviso, 0)} pontos")
print(f"  → Com uma instrução de cuidado, cai para {fmt(prompt_cuidadoso*100, 0)}% dos resumos")
print("\n  CONTA 6 · O telefone sem fio (modelo central)")
print(f"  → No resumo dos autores:              {fmt(pi_central*100)}% [{fmt(ic_pi[0]*100)}; {fmt(ic_pi[1]*100)}]")
print(f"  → Depois do resumo de uma IA:         {fmt(ic_cadeia[1]*100)}% [{fmt(ic_cadeia[0]*100)}; {fmt(ic_cadeia[2]*100)}]")
print(f"  → IA com instrução de cuidado:        {fmt(ic_cuidado[1]*100)}% [{fmt(ic_cuidado[0]*100)}; {fmt(ic_cuidado[2]*100)}]")
print(f"  → P(mais de 80% chegam como causa):   {'acima de 99%' if p_cadeia_maior_80 > 0.995 else fmt(p_cadeia_maior_80*100, 0) + '%'}")
print(f"  → Dos que chegam como causa, a IA acrescentou a causa em {fmt(frac_acrescentada[1]*100, 0)}% [{fmt(frac_acrescentada[0]*100, 0)}; {fmt(frac_acrescentada[2]*100, 0)}]")
print(f"  → Conferência com 100 mil estudos simulados um a um: {fmt(direto*100)}%")
print("=" * 72)

In [ ]:
# --- VISUALIZAÇÃO ---
RODAPE = 'Fonte: Isch, Dörr, Fasching, Jennings & Watts (2026), Nature Human Behaviour. Conta própria | #365Probabilidades'

# GRÁFICO 1 · Quase metade
fig1, ax1 = plt.subplots(figsize=(12, 8))
rot = ['Como o classificador\nmarcou', 'Corrigido pelos erros\ndo classificador']
vals = [p_causal * 100, pi_central * 100]
erros = [[0, pi_central * 100 - ic_pi[0] * 100], [0, ic_pi[1] * 100 - pi_central * 100]]
b1 = ax1.bar(rot, vals, color=[DOURADO, VERDE], width=0.45, yerr=erros, capsize=8)
for b, v in zip(b1, vals):
    ax1.text(b.get_x() + b.get_width()/2, v + 6, f"{fmt(v)}%", ha='center', fontsize=18, fontweight='bold')
ax1.axhline(50, color=CINZA, linestyle='--', linewidth=1.2); ax1.text(-0.4, 51, 'metade', fontsize=11, color=CINZA)
ax1.set_ylim(0, 70); ax1.set_ylabel('Artigos transversais com linguagem de causa (%)'); ax1.tick_params(axis='x', labelsize=13)
ax1.set_title('Quase metade dos estudos que só observam fala como se tivesse testado causa\n194.631 artigos de ciências sociais, 1980 a 2024', fontsize=14, pad=15)
plt.figtext(0.5, 0.005, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-167-grafico-01-quase-metade.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# GRÁFICO 2 · Onde mais
fig2, ax2 = plt.subplots(figsize=(12, 8))
rot2 = ['Até 2000', 'Em 2024', 'Revistas abaixo da\nmediana de impacto', 'Revistas acima da\nmediana de impacto']
v2 = [tendencia['até 2000'] * 100, tendencia['2024'] * 100, revistas['abaixo'] * 100, revistas['acima'] * 100]
b2 = ax2.bar(rot2, v2, color=[CINZA, VERMELHO, CINZA, VERMELHO], width=0.55)
for b, v, prefixo in zip(b2, v2, ['cerca de ', 'mais de ', '', '']):
    ax2.text(b.get_x() + b.get_width()/2, v + 1.5, f"{prefixo}{fmt(v, 0)}%", ha='center', fontsize=14, fontweight='bold')
ax2.axvline(1.5, color=CINZA, linewidth=1)
ax2.set_ylim(0, 75); ax2.set_ylabel('Artigos transversais com linguagem de causa (%)'); ax2.tick_params(axis='x', labelsize=12)
ax2.set_title('O exagero triplicou desde 2000 e é maior nas revistas de mais impacto\nPelo tempo e pelo prestígio da revista', fontsize=14, pad=15)
plt.figtext(0.5, 0.005, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-167-grafico-02-onde-mais.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# GRÁFICO 3 · Quando a IA resume
fig3, ax3 = plt.subplots(figsize=(12, 8))
rot3 = list(resumo_ia) + ['Recebeu uma instrução de cuidado\n(cinco modelos)']
v3 = [v * 100 for v in resumo_ia.values()] + [prompt_cuidadoso * 100]
ys = np.arange(len(rot3))[::-1]
cores3 = [VERMELHO, VERMELHO, DOURADO, DOURADO, VERDE]
ax3.barh(ys, v3, color=cores3, height=0.55)
for y, v in zip(ys, v3):
    ax3.text(v + 1, y, f"{fmt(v, 0)}%", va='center', fontsize=14, fontweight='bold')
ax3.set_yticks(ys); ax3.set_yticklabels([r.replace(' + ', '\n+ ').replace('resumo reescrito', 'resumo\nreescrito') for r in rot3], fontsize=12); ax3.set_xlim(0, 112)
ax3.set_xlabel('Resumos feitos por IA com linguagem de causa (%)')
ax3.set_title('A IA repete o exagero, a não ser que alguém avise\nResumos de artigos transversais feitos por modelos de linguagem', fontsize=14, pad=15)
plt.figtext(0.5, 0.005, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-167-grafico-03-quando-a-ia-resume.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# GRÁFICO 4 · O caminho (árvore de probabilidades)
fig4, ax4 = plt.subplots(figsize=(12, 8))
ax4.set_xlim(0, 12); ax4.set_ylim(0, 8); ax4.axis('off')
def caixa(x, y, txt, cor, w=2.9, h=1.25, fs=12):
    ax4.add_patch(plt.Rectangle((x - w/2, y - h/2), w, h, color=cor, alpha=0.9))
    ax4.text(x, y, txt, ha='center', va='center', color='white', fontsize=fs, fontweight='bold')
def seta(x0, y0, x1, y1, rot):
    ax4.annotate('', xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle='->', color='#333', lw=1.8))
    dy = 0.3 if y1 >= y0 else -0.35
    ax4.text((x0 + x1)/2 - (0.35 if y1 < y0 else 0), (y0 + y1)/2 + dy, rot, ha='center', fontsize=11, family='monospace',
             bbox=dict(facecolor='white', edgecolor='none', pad=1))
q1, q0 = resumo_ia['Recebeu o resumo original'], resumo_ia['Recebeu o resumo reescrito sem causa']
caixa(1.6, 4, 'Estudo que só\nviu as coisas\njuntas', CINZA)
caixa(5.6, 6, 'Resumo diz\n"causa"', VERMELHO)
caixa(5.6, 2, 'Resumo não\ndiz "causa"', VERDE)
caixa(10.2, 7.1, 'IA diz "causa"', VERMELHO, h=0.9)
caixa(10.2, 5.2, 'IA não diz', VERDE, h=0.9)
caixa(10.2, 2.9, 'IA diz "causa"', VERMELHO, h=0.9)
caixa(10.2, 1.0, 'IA não diz', VERDE, h=0.9)
seta(3.05, 4.4, 4.15, 5.7, f'π = {fmt(pi_central*100,0)}%')
seta(3.05, 3.6, 4.15, 2.3, f'1 − π = {fmt((1-pi_central)*100,0)}%')
seta(7.05, 6.2, 8.75, 7.0, f'q₁ = {fmt(q1*100,0)}%')
seta(7.05, 5.8, 8.75, 5.3, f'{fmt((1-q1)*100,0)}%')
seta(7.05, 2.2, 8.75, 2.9, f'q₀ = {fmt(q0*100,0)}%')
seta(7.05, 1.8, 8.75, 1.1, f'{fmt((1-q0)*100,0)}%')
ax4.text(6, 0.05, f'P(chega como causa) = π·q₁ + (1 − π)·q₀ = {fmt(ic_cadeia[1]*100)}%   IC 95% [{fmt(ic_cadeia[0]*100)}; {fmt(ic_cadeia[2]*100)}]',
         ha='center', fontsize=13, family='monospace', color=VERMELHO, fontweight='bold')
ax4.set_title('O telefone sem fio de um estudo\nDo resumo dos autores ao resumo de uma IA', fontsize=14, pad=15)
plt.figtext(0.5, 0.005, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-167-grafico-04-o-caminho.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")

# GRÁFICO 5 · As três distribuições
fig5, ax5 = plt.subplots(figsize=(12, 8))
cenarios = [('IA com instrução de cuidado', cuidado_sim, VERDE),
            ('No resumo dos autores', pi_sim, DOURADO),
            ('Depois do resumo de uma IA', cadeia_ia, VERMELHO)]
for nome, sim, cor in cenarios:
    ax5.hist(sim * 100, bins=120, density=True, color=cor, alpha=0.75, label=nome)
    m = np.median(sim) * 100
    ax5.axvline(m, color=cor, lw=2)
    ax5.text(m, ax5.get_ylim()[1] * 0.02 if False else 0, '', fontsize=1)
for nome, sim, cor in cenarios:
    lo, m, hi = np.percentile(sim * 100, [2.5, 50, 97.5])
    ax5.annotate(f'{fmt(m,0)}%\n[{fmt(lo,0)}; {fmt(hi,0)}]', xy=(m, 0), xytext=(m, ax5.get_ylim()[1] * 0.85),
                 ha='center', fontsize=13, fontweight='bold', color=cor,
                 bbox=dict(facecolor='white', edgecolor='none', pad=2))
ax5.set_xlim(0, 100)
ax5.set_xlabel('Probabilidade de um estudo que só viu as coisas juntas chegar até você dizendo "causa" (%)', fontsize=12)
ax5.set_ylabel('Densidade (20 mil simulações)')
ax5.legend(loc='upper left', fontsize=11, frameon=False, bbox_to_anchor=(0.0, 0.75))
ax5.set_title('Metade sai do estudo dizendo "causa". Depois da IA, quase todos\nMonte Carlo com IC 95%', fontsize=14, pad=15)
plt.figtext(0.5, 0.005, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-167-grafico-05-tres-cenarios.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 5 salvo!")

### 💡 O Insight

**Um estudo que só viu as coisas juntas sai dizendo "causa" em 47% das vezes. Depois que uma IA resume, chega até você dizendo "causa" em 93% das vezes (IC 95% de 85% a 98%).**

O primeiro número vem de 194.631 estudos de ciências sociais, já descontados os erros da máquina que fez a contagem. Quase metade diz que uma coisa provoca outra sem ter um desenho que permita dizer isso.

O segundo vem do modelo do dia, que simula o caminho do estudo até quem lê. Metade dos estudos que chegam dizendo "causa" não dizia isso no resumo. Foi a IA que acrescentou.

Quando a IA recebe uma instrução de cuidado, a probabilidade cai para 15% (IC 95% de 9% a 23%).

O exagero triplicou desde 2000, de cerca de 20% para mais de 60% em 2024. E é maior nas revistas de mais impacto, 54% contra 43%. As revistas de mais impacto não são as mais cuidadosas.

O exagero começa no próprio estudo, e a IA, sem instrução, aumenta. Com um aviso sobre o desenho do estudo, a linguagem de causa dos resumos da IA caiu de 99% para 42%.

É por isso que aqui, todo dia, eu digo que tipo de estudo é e o que ele pode ou não afirmar.

A pergunta que fica:

*Da última vez que você leu "segundo um estudo", você foi ver como o estudo foi feito?*

---

### ⚠️ Limitações do Modelo

- A contagem foi feita por classificadores automáticos, validados contra codificação humana. O classificador final acertou cerca de 84% das vezes. A Conta 2 corrige a proporção por esse erro, supondo que os acertos medidos no teste valem para todo o conjunto.
- A especificidade do classificador não foi publicada. Ela foi deduzida da precisão, da sensibilidade e da proporção observada.
- A análise olha o título e o resumo, e não o texto inteiro. No texto inteiro, a maior parte das afirmações é associativa (73%), e só 15,5% são causais sem ressalva.
- Linguagem de causa não prova má-fé. Parte dela pode ser hábito da área ou escrita pouco precisa. E alguns estudos transversais usam métodos que sustentam conclusões mais fortes.
- Os números do tempo (cerca de 20% até 2000, mais de 60% em 2024) vêm do texto do artigo, que não traz os valores ano a ano.
- A Conta 3 supõe metade dos 127.598 artigos de cada lado da mediana de impacto.
- Os percentuais dos resumos feitos por IA vêm de um só modelo no primeiro experimento. A instrução de cuidado reduziu a linguagem de causa em todos os modelos, menos no mesmo modelo do primeiro experimento.
- Os artigos analisados são de cinco áreas das ciências sociais, em bases de dados em inglês.
- No modelo do telefone sem fio, a chance de a IA pôr causa num resumo que não diz causa (q₀ = 90%) vem da condição em que os autores reescreveram resumos sem linguagem de causa. Ela é usada como aproximação para os resumos que já nascem sem causa.
- q₁ e q₀ vêm de um só modelo (GPT-4.1) e de 28 artigos de revistas de elite. A incerteza foi contada por artigo, e não pelos 280 resumos, porque os dez resumos de um mesmo artigo não são independentes. Isso deixa o intervalo mais largo, e mais honesto.
- O cenário com instrução de cuidado usa a média de cinco modelos em 100 artigos, de outro experimento do mesmo artigo. Os valores por modelo não estão no texto.
- O último elo, o leitor, não entra na conta. O artigo mostra que, em todas as condições, a maioria dos leitores usou linguagem de causa no próprio resumo, mas não publica os percentuais.
- O modelo supõe que alguém leu o estudo por meio de um resumo de IA. Quem lê o resumo original fica no primeiro número, 47%.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*